In [1]:
!pip install -q streamlit pyjwt pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 100.5 MB/s eta 0:00:00


In [12]:

%%writefile app.py
import streamlit as st
import sqlite3
import hashlib
import re
import random
import smtplib
import os
import secrets
from email.mime.text import MIMEText
import jwt
from datetime import datetime, timedelta, timezone
from contextlib import closing

# ---------- CONFIG ----------
JWT_SECRET = os.environ.get("JWT_SECRET")
EMAIL_ADDRESS = os.environ.get("EMAIL_ID")
EMAIL_PASSWORD = os.environ.get("EMAIL_PASSWORD")

ADMIN_USER = "admin"
ADMIN_PASS = "Admin@123"

DB = "users.db"
OTP_EXPIRY_SECONDS = 300       # 5 minutes
MAX_OTP_ATTEMPTS = 5


# ---------- DATABASE ----------
def get_db():
    conn = sqlite3.connect(DB, timeout=10)
    conn.execute("PRAGMA busy_timeout = 10000")
    return conn


def init_db():
    with closing(get_db()) as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS users (
                username TEXT PRIMARY KEY,
                email TEXT UNIQUE,
                password TEXT,
                question TEXT,
                answer TEXT
            )
        """)
        conn.commit()


init_db()


# ---------- PASSWORD HASHING ----------
# SHA-256 is retained so existing accounts remain compatible.
def hash_pw(password):
    return hashlib.sha256(password.encode("utf-8")).hexdigest()


# ---------- VALIDATION ----------
def valid_email(email):
    return bool(re.fullmatch(
        r"[A-Za-z0-9._%+-]{2,}@[A-Za-z0-9-]{2,}\.[A-Za-z]{2,}",
        email.strip()
    ))


def valid_pw(password):
    return (
        len(password) >= 8
        and bool(re.search(r"[A-Z]", password))
        and bool(re.search(r"[a-z]", password))
        and bool(re.search(r"[0-9]", password))
        and bool(re.search(r"[^A-Za-z0-9]", password))
    )


# ---------- JWT ----------
def make_token(username):
    if not JWT_SECRET:
        raise RuntimeError("JWT_SECRET is not configured.")

    payload = {
        "user": username,
        "exp": datetime.now(timezone.utc) + timedelta(hours=2)
    }

    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")


def verify_token(token):
    if not token or not JWT_SECRET:
        return None

    try:
        payload = jwt.decode(
            token,
            JWT_SECRET,
            algorithms=["HS256"]
        )
        return payload.get("user")
    except jwt.InvalidTokenError:
        return None
    except Exception:
        return None


# ---------- EMAIL OTP ----------
def send_otp(to_email, otp):
    if not EMAIL_ADDRESS or not EMAIL_PASSWORD:
        raise RuntimeError("Email configuration is missing.")

    msg = MIMEText(
        f"Your password reset OTP is: {otp}\n\n"
        f"This OTP expires in 5 minutes. "
        f"Do not share it with anyone."
    )

    msg["Subject"] = "Password Reset OTP"
    msg["From"] = EMAIL_ADDRESS
    msg["To"] = to_email

    with smtplib.SMTP_SSL("smtp.gmail.com", 465, timeout=20) as server:
        server.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
        server.send_message(msg)


# ---------- SESSION STATE ----------
st.set_page_config(
    page_title="Auth App",
    page_icon="🔐",
    layout="centered"
)

defaults = {
    "token": None,
    "page": "Login",
    "otp": None,
    "otp_email": None,
    "otp_created_at": None,
    "otp_attempts": 0,
    "admin": False,
    "admin_notice": None,
    "login_notice": None
}

for key, value in defaults.items():
    if key not in st.session_state:
        st.session_state[key] = value


def clear_otp():
    st.session_state.otp = None
    st.session_state.otp_email = None
    st.session_state.otp_created_at = None
    st.session_state.otp_attempts = 0


def go_to_login():
    st.session_state.page = "Login"
    st.session_state.admin = False
    st.session_state.token = None
    st.rerun()


# ---------- USER DASHBOARD ----------
user = verify_token(st.session_state.token)

if st.session_state.token and not user:
    st.session_state.token = None
    st.session_state.login_notice = (
        "Your session expired. Please log in again."
    )

if user:
    st.title(f"👋 Welcome, {user}")
    st.success("You are logged in.")

    if st.button("Logout", use_container_width=True):
        go_to_login()

    st.stop()


# ---------- ADMIN DASHBOARD ----------
if st.session_state.page == "Admin":

    # Always provide a way to leave the Admin page.
    if st.button("⬅️ Back to Login", use_container_width=True):
        go_to_login()

    st.title("🛡️ Admin Dashboard")

    if st.session_state.admin:
        st.success("Admin login successful.")

        try:
            with closing(get_db()) as conn:
                rows = conn.execute(
                    "SELECT username, email FROM users ORDER BY username"
                ).fetchall()

            st.metric("Total registered users", len(rows))

            if rows:
                st.dataframe(
                    [
                        {"Username": row[0], "Email": row[1]}
                        for row in rows
                    ],
                    use_container_width=True,
                    hide_index=True
                )
            else:
                st.info("No registered users found.")

        except sqlite3.Error:
            st.error("Unable to read the user database.")

        if st.button("Admin Logout", use_container_width=True):
            st.session_state.admin = False
            st.session_state.admin_notice = "Admin logged out."
            go_to_login()

    else:
        st.info("Enter your administrator credentials.")

        admin_username = st.text_input(
            "Admin Username",
            key="admin_username"
        )
        admin_password = st.text_input(
            "Admin Password",
            type="password",
            key="admin_password"
        )

        if st.button("Admin Login", use_container_width=True):
            if not admin_username.strip() or not admin_password:
                st.error("Please enter both admin username and password.")

            elif (
                secrets.compare_digest(
                    admin_username.strip(), ADMIN_USER
                )
                and secrets.compare_digest(
                    admin_password, ADMIN_PASS
                )
            ):
                st.session_state.admin = True
                st.rerun()

            else:
                st.error(
                    "Invalid admin credentials. Check the admin "
                    "username and password."
                )

    st.stop()


# ---------- NAVIGATION ----------
pages = ["Login", "Signup", "Forgot Password", "Admin"]

if st.session_state.page not in pages:
    st.session_state.page = "Login"

choice = st.sidebar.radio(
    "Navigate",
    pages,
    index=pages.index(st.session_state.page),
    key="main_navigation"
)

st.session_state.page = choice

if st.session_state.admin:
    st.session_state.admin = False

if st.session_state.admin_notice:
    st.info(st.session_state.admin_notice)
    st.session_state.admin_notice = None

if st.session_state.login_notice:
    st.info(st.session_state.login_notice)
    st.session_state.login_notice = None


# ---------- LOGIN ----------
if choice == "Login":
    st.title("🔐 Login")

    username_or_email = st.text_input(
        "Username or Email",
        key="login_username"
    ).strip()

    password = st.text_input(
        "Password",
        type="password",
        key="login_password"
    )

    if st.button("Login", use_container_width=True):
        if not username_or_email or not password:
            st.error("All fields are required.")

        else:
            try:
                with closing(get_db()) as conn:
                    row = conn.execute(
                        """
                        SELECT username, password
                        FROM users
                        WHERE username = ? OR email = ?
                        """,
                        (
                            username_or_email,
                            username_or_email
                        )
                    ).fetchone()

                if row and secrets.compare_digest(
                    row[1], hash_pw(password)
                ):
                    st.session_state.token = make_token(row[0])
                    st.rerun()

                else:
                    st.error(
                        "Invalid username/email or password. "
                        "If you have not registered, please sign up."
                    )

            except sqlite3.Error:
                st.error("Unable to access the user database.")

            except RuntimeError:
                st.error(
                    "Login configuration is incomplete. "
                    "Please contact the administrator."
                )


# ---------- SIGNUP ----------
elif choice == "Signup":
    st.title("📝 Signup")

    username = st.text_input("Username", key="signup_username").strip()
    email = st.text_input("Email", key="signup_email").strip().lower()
    password = st.text_input(
        "Password", type="password", key="signup_password"
    )
    confirm_password = st.text_input(
        "Confirm Password", type="password", key="signup_confirm"
    )

    question = st.selectbox(
        "Security Question",
        [
            "What is your pet name?",
            "What is your birth city?",
            "What is your favorite color?"
        ]
    )

    answer = st.text_input(
        "Security Answer",
        key="signup_answer"
    )

    if st.button("Sign Up", use_container_width=True):
        if not all([
            username, email, password,
            confirm_password, question, answer.strip()
        ]):
            st.error("All fields are required.")

        elif not valid_email(email):
            st.error("Invalid email address.")

        elif not valid_pw(password):
            st.error(
                "Password needs 8+ characters, an uppercase letter, "
                "a lowercase letter, a digit, and a symbol."
            )

        elif password != confirm_password:
            st.error("Passwords do not match.")

        else:
            try:
                with closing(get_db()) as conn:
                    conn.execute(
                        """
                        INSERT INTO users
                        (username, email, password, question, answer)
                        VALUES (?, ?, ?, ?, ?)
                        """,
                        (
                            username,
                            email,
                            hash_pw(password),
                            question,
                            hash_pw(answer.strip().lower())
                        )
                    )
                    conn.commit()

                st.success(
                    "Signup successful! Please go to Login."
                )

            except sqlite3.IntegrityError:
                st.error(
                    "Username or email already exists. "
                    "Try logging in instead."
                )

            except sqlite3.Error:
                st.error("Unable to save your account.")


# ---------- FORGOT PASSWORD ----------
elif choice == "Forgot Password":
    st.title("🔑 Forgot Password")

    route = st.radio(
        "Recovery method",
        ["Security Question", "Email OTP"],
        key="recovery_method"
    )

    # ----- SECURITY QUESTION -----
    if route == "Security Question":
        username = st.text_input(
            "Username", key="recovery_username"
        ).strip()

        if username:
            try:
                with closing(get_db()) as conn:
                    row = conn.execute(
                        "SELECT question FROM users WHERE username = ?",
                        (username,)
                    ).fetchone()

                if row:
                    st.info("Question: " + row[0])

                    answer = st.text_input(
                        "Answer",
                        type="password",
                        key="security_answer"
                    )
                    new_password = st.text_input(
                        "New Password",
                        type="password",
                        key="security_new_password"
                    )
                    confirm_new_password = st.text_input(
                        "Confirm New Password",
                        type="password",
                        key="security_confirm_password"
                    )

                    if st.button(
                        "Reset Password",
                        use_container_width=True
                    ):
                        if not answer or not new_password or not confirm_new_password:
                            st.error("All fields are required.")

                        elif not valid_pw(new_password):
                            st.error("Weak password. Please use the required format.")

                        elif new_password != confirm_new_password:
                            st.error("Passwords do not match.")

                        else:
                            with closing(get_db()) as conn:
                                saved = conn.execute(
                                    "SELECT answer FROM users WHERE username = ?",
                                    (username,)
                                ).fetchone()

                                if (
                                    saved
                                    and secrets.compare_digest(
                                        saved[0],
                                        hash_pw(answer.strip().lower())
                                    )
                                ):
                                    conn.execute(
                                        """
                                        UPDATE users
                                        SET password = ?
                                        WHERE username = ?
                                        """,
                                        (hash_pw(new_password), username)
                                    )
                                    conn.commit()
                                    st.success(
                                        "Password reset successfully. "
                                        "Please log in."
                                    )
                                else:
                                    st.error("Wrong security answer.")

                else:
                    st.warning("User not found.")

            except sqlite3.Error:
                st.error("Unable to access the user database.")

    # ----- EMAIL OTP -----
    else:
        email = st.text_input(
            "Registered Email",
            key="otp_registered_email"
        ).strip().lower()

        if st.button("Send OTP", use_container_width=True):
            clear_otp()

            if not email:
                st.error("Please enter your registered email.")

            else:
                try:
                    with closing(get_db()) as conn:
                        row = conn.execute(
                            "SELECT email FROM users WHERE email = ?",
                            (email,)
                        ).fetchone()

                    if not row:
                        st.error("Email not registered.")

                    else:
                        otp = f"{random.SystemRandom().randint(0, 999999):06d}"

                        try:
                            send_otp(email, otp)

                            st.session_state.otp = otp
                            st.session_state.otp_email = email
                            st.session_state.otp_created_at = (
                                datetime.now(timezone.utc).timestamp()
                            )
                            st.session_state.otp_attempts = 0

                            st.success(
                                "OTP sent successfully. "
                                "It expires in 5 minutes."
                            )

                        except Exception:
                            clear_otp()
                            st.error(
                                "Unable to send OTP. Please check the "
                                "email configuration and try again."
                            )

                except sqlite3.Error:
                    st.error("Unable to access the user database.")

        if st.session_state.otp and st.session_state.otp_email:
            elapsed = (
                datetime.now(timezone.utc).timestamp()
                - st.session_state.otp_created_at
            )

            if elapsed >= OTP_EXPIRY_SECONDS:
                clear_otp()
                st.warning("OTP expired. Please request a new OTP.")

            elif st.session_state.otp_attempts >= MAX_OTP_ATTEMPTS:
                clear_otp()
                st.error(
                    "Maximum OTP attempts reached. "
                    "Please request a new OTP."
                )

            else:
                st.caption("Enter the OTP sent to your registered email.")

                code = st.text_input(
                    "Enter OTP",
                    max_chars=6,
                    key="otp_code"
                ).strip()

                new_password = st.text_input(
                    "New Password",
                    type="password",
                    key="np2"
                )

                confirm_new_password = st.text_input(
                    "Confirm New Password",
                    type="password",
                    key="cp2"
                )

                if st.button(
                    "Verify & Reset",
                    use_container_width=True
                ):
                    # Check expiry again when the button is clicked.
                    elapsed = (
                        datetime.now(timezone.utc).timestamp()
                        - st.session_state.otp_created_at
                    )

                    if elapsed >= OTP_EXPIRY_SECONDS:
                        clear_otp()
                        st.error("OTP expired. Please request a new OTP.")

                    elif st.session_state.otp_attempts >= MAX_OTP_ATTEMPTS:
                        clear_otp()
                        st.error(
                            "Maximum OTP attempts reached. "
                            "Please request a new OTP."
                        )

                    elif not code or not new_password or not confirm_new_password:
                        st.error("All fields are required.")

                    elif not secrets.compare_digest(
                        code, st.session_state.otp
                    ):
                        st.session_state.otp_attempts += 1
                        remaining = (
                            MAX_OTP_ATTEMPTS
                            - st.session_state.otp_attempts
                        )

                        if remaining <= 0:
                            clear_otp()
                            st.error(
                                "Maximum OTP attempts reached. "
                                "Please request a new OTP."
                            )
                        else:
                            st.error(
                                f"Incorrect OTP. {remaining} "
                                "attempts remaining."
                            )

                    elif not valid_pw(new_password):
                        st.error(
                            "Weak password. Use 8+ characters with "
                            "uppercase, lowercase, digit, and symbol."
                        )

                    elif new_password != confirm_new_password:
                        st.error("Passwords do not match.")

                    else:
                        try:
                            with closing(get_db()) as conn:
                                result = conn.execute(
                                    """
                                    UPDATE users
                                    SET password = ?
                                    WHERE email = ?
                                    """,
                                    (
                                        hash_pw(new_password),
                                        st.session_state.otp_email
                                    )
                                )
                                conn.commit()

                                if result.rowcount == 1:
                                    clear_otp()
                                    st.success(
                                        "Password reset successfully. "
                                        "Please log in."
                                    )
                                else:
                                    clear_otp()
                                    st.error(
                                        "Account not found. "
                                        "Please request a new OTP."
                                    )

                        except sqlite3.Error:
                            st.error(
                                "Unable to update the password. "
                                "Please try again."
                            )


Overwriting app.py


In [13]:
import subprocess, time, os
from pyngrok import ngrok, conf
from google.colab import userdata

# 1) Read secrets IN THE NOTEBOOK (works here)
JWT_SECRET     = userdata.get("JWT_SECRET")
NGROK_TOKEN    = userdata.get("NGROK_AUTHTOKEN")
EMAIL_ID       = userdata.get("EMAIL_ID")
EMAIL_PASSWORD = userdata.get("EMAIL_PASSWORD")

# 2) Sanity check
for name, val in [("JWT_SECRET", JWT_SECRET),
                  ("NGROK_AUTHTOKEN", NGROK_TOKEN),
                  ("EMAIL_ID", EMAIL_ID),
                  ("EMAIL_PASSWORD", EMAIL_PASSWORD)]:
    if not val:
        raise Exception(f"❌ Secret {name} is missing or Notebook access is OFF")
    print(f"✅ {name} loaded (len={len(val)})")

# 3) Kill old tunnels/processes
try: ngrok.kill()
except: pass
os.system("pkill -f streamlit || true")
time.sleep(3)

# 4) Configure ngrok
conf.get_default().auth_token = NGROK_TOKEN

# 5) Build env dict to pass to Streamlit subprocess
env = os.environ.copy()
env["JWT_SECRET"]     = JWT_SECRET
env["EMAIL_ID"]       = EMAIL_ID
env["EMAIL_PASSWORD"] = EMAIL_PASSWORD

# 6) Launch Streamlit WITH the env vars
proc = subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port", "8501",
    "--server.headless", "true",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false"
], env=env)

print("⏳ Booting Streamlit...")
time.sleep(10)

# 7) Open ngrok tunnel
url = ngrok.connect(8501, "http", pooling_enabled=True).public_url
print("=" * 60)
print("🚀 OPEN THIS URL:", url)
print("=" * 60)

✅ JWT_SECRET loaded (len=8)
✅ NGROK_AUTHTOKEN loaded (len=49)
✅ EMAIL_ID loaded (len=20)
✅ EMAIL_PASSWORD loaded (len=19)
⏳ Booting Streamlit...
🚀 OPEN THIS URL: https://unwilling-skewer-imaginary.ngrok-free.dev
